# Part 2 — Deep Dive Analysis

MNIST CNN experiments: architecture comparison, data augmentation, regularization, and hyperparameter tuning.

All results loaded from the `outputs/` directory generated by the training scripts.

---
**Experiments run:**
- 5 CNN architectures (capacity comparison)
- 2 augmentation variants (with vs. without)
- 6 regularization configurations
- 18 hyperparameter tuning configurations

**Best overall result:** `tune_07_augmented` — **99.39% test accuracy** on MNIST

In [ ]:
import json
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import Image, display
import warnings
warnings.filterwarnings('ignore')

NOTEBOOK_DIR = Path(r'C:\Users\emil_\vscode\Assignment1\part_2')
OUTPUTS = NOTEBOOK_DIR / 'outputs'

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
PALETTE = list(plt.cm.tab10.colors)

def load_json(path):
    with open(path) as f:
        return json.load(f)

def load_run(run_dir):
    run_dir = Path(run_dir)
    result = {'name': run_dir.name, 'path': run_dir}
    for fname in ('summary.json', 'config.json', 'training_history.json'):
        fpath = run_dir / fname
        if fpath.exists():
            key = fname.replace('.json', '')
            result[key] = load_json(fpath)
    return result

def load_group(group_dir):
    return {d.name: load_run(d) for d in sorted(Path(group_dir).iterdir()) if d.is_dir()}

cnn_exps   = load_group(OUTPUTS / 'cnn_comparison_2026-04-28_154826')
aug_exps   = load_group(OUTPUTS / 'augmentation_comparison_2026-04-28_154632')
reg_exps   = load_group(OUTPUTS / 'regularization_comparison_2026-04-28_155225')
tune_exps  = load_group(OUTPUTS / 'hyperparameter_tuning_2026-04-28_155935')

print(f'CNN architectures : {len(cnn_exps)}')
print(f'Augmentation runs : {len(aug_exps)}')
print(f'Regularization    : {len(reg_exps)}')
print(f'Tuning runs       : {len(tune_exps)}')

---
## 1. CNN Architecture Comparison

Five architectures tested at matched or similar parameter counts.  
All trained on identical data splits (seed=42), 5 epochs, LR=0.001, Adam, no augmentation.

In [ ]:
# --- Architecture summary table ---
arch_order = ['cnn_small', 'cnn_medium', 'cnn_dropout', 'cnn_deep_balanced', 'cnn_deep_wide']
arch_labels = ['Small\n(2-conv\n16/32)', 'Medium\n(2-conv\n32/64)', 'Dropout\n(2-conv\n32/64)',
               'Deep-Bal\n(3-conv\n32/64/64)', 'Deep-Wide\n(3-conv\n32/64/128)']

accs   = [cnn_exps[n]['summary']['final_test_accuracy'] * 100 for n in arch_order]
params = [cnn_exps[n]['config']['trainable_parameters'] for n in arch_order]
times  = [cnn_exps[n]['summary']['total_training_time_seconds'] for n in arch_order]

print(f'{'Architecture':<18} {'Params':>8} {'Test Acc':>9} {'Train Time':>12}')
print('-' * 52)
for name, p, a, t in zip(arch_order, params, accs, times):
    print(f'{name:<18} {p:>8,} {a:>8.2f}% {t:>10.1f}s')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Test accuracy bar
ax = axes[0]
bars = ax.bar(arch_labels, accs, color=PALETTE[:5], edgecolor='black', linewidth=0.5)
ax.set_ylim(98.5, 99.5)
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Test Accuracy by Architecture')
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{acc:.2f}%', ha='center', va='bottom', fontsize=8)

# Params vs accuracy scatter
ax = axes[1]
for i, (p, a, lab) in enumerate(zip(params, accs, arch_labels)):
    ax.scatter(p, a, color=PALETTE[i], s=160, edgecolors='black', linewidths=0.5, zorder=5)
    ax.annotate(lab.split('\n')[0], (p, a), textcoords='offset points',
                xytext=(5, 3), fontsize=8)
ax.set_xlabel('Trainable Parameters')
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Accuracy vs. Model Size')
ax.set_ylim(98.5, 99.5)

# Training time bar
ax = axes[2]
bars = ax.bar(arch_labels, times, color=PALETTE[:5], edgecolor='black', linewidth=0.5)
ax.set_ylabel('Total Training Time (s)')
ax.set_title('Training Time by Architecture')
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{t:.0f}s', ha='center', va='bottom', fontsize=8)

plt.suptitle('Section 1: CNN Architecture Comparison (MNIST, 5 epochs)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUTS / 'deep_dive_cnn_comparison.png', bbox_inches='tight', dpi=120)
plt.show()

### Architecture Analysis

**Key findings:**

1. **Deeper beats wider at the same scale.** `cnn_deep_wide` (3-conv, 32/64/128) reaches 99.26% with 390K params — outperforming `cnn_medium` (2-conv, 32/64) at 99.11% with 421K params. More layers ≠ more parameters, but adds representational depth.

2. **Capacity has diminishing returns.** Going from Small (105K) to Medium (421K) gains ~0.31%, but the jump is not linear with parameter count.

3. **Dropout alone doesn't improve accuracy.** `cnn_dropout` (same arch as medium, dropout=0.3) scores 99.03% vs medium's 99.11%. Dropout adds training regularization but may slow convergence in early epochs — more visible with short training runs.

4. **3 conv layers detect more useful features.** The hierarchical feature abstraction benefit of an extra conv layer is visible: `cnn_deep_balanced` (99.14%) and `cnn_deep_wide` (99.26%) both beat their 2-layer equivalents.

**What do different conv layers detect?**
- **Layer 1** (closest to input): Low-level features — edges, gradients, simple strokes. Kernels often resemble Gabor filters (oriented edge detectors).
- **Layer 2**: Mid-level features — curves, corners, stroke intersections, small digit components.
- **Layer 3** (deepest): High-level features — digit parts, loops, endpoints — patterns that are uniquely discriminative per class.

Translation invariance comes from the max-pooling layers after each conv: the *where* is discarded, keeping only *what was detected*.

In [ ]:
# Show conv filter visualizations from best architecture (cnn_deep_wide)
filter_first = OUTPUTS / 'cnn_comparison_2026-04-28_154826' / 'cnn_deep_wide' / 'conv_filters_first.png'
filter_last  = OUTPUTS / 'cnn_comparison_2026-04-28_154826' / 'cnn_deep_wide' / 'conv_filters_last.png'

if filter_first.exists() and filter_last.exists():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(plt.imread(str(filter_first)))
    axes[0].set_title('First Conv Layer Filters\n(detect edges & low-level strokes)', fontsize=11)
    axes[0].axis('off')
    axes[1].imshow(plt.imread(str(filter_last)))
    axes[1].set_title('Last Conv Layer Filters\n(detect high-level digit components)', fontsize=11)
    axes[1].axis('off')
    plt.suptitle('Learned Convolutional Filters — cnn_deep_wide', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print('Filter images not found. Run cnn_comparison.py to generate.')

---
## 2. Data Augmentation Analysis

Augmentation applied: `RandomAffine(rotation=±10°, translate=10%, scale=0.95–1.05)`  
Hypothesis: augmentation prevents overfitting by exposing the model to geometrically diverse variants of each digit.

In [ ]:
aug_with    = aug_exps['with_augmentation']
aug_without = aug_exps['without_augmentation']

# Load training histories
hist_with    = aug_with['training_history']
hist_without = aug_without['training_history']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy curves
ax = axes[0]
epochs_w  = [e['epoch'] for e in hist_with]
epochs_wo = [e['epoch'] for e in hist_without]
val_w  = [e['val_accuracy'] * 100 for e in hist_with]
val_wo = [e['val_accuracy'] * 100 for e in hist_without]
train_w  = [e['train_accuracy'] * 100 for e in hist_with]
train_wo = [e['train_accuracy'] * 100 for e in hist_without]

ax.plot(epochs_w,  train_w,  color='steelblue', linestyle='--', label='Train (aug)', linewidth=1.5)
ax.plot(epochs_w,  val_w,    color='steelblue', linestyle='-',  label='Val (aug)', linewidth=2)
ax.plot(epochs_wo, train_wo, color='coral', linestyle='--', label='Train (no aug)', linewidth=1.5)
ax.plot(epochs_wo, val_wo,   color='coral', linestyle='-',  label='Val (no aug)', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Training vs Validation Accuracy')
ax.legend(fontsize=9)

# Loss curves
ax = axes[1]
loss_w  = [e['val_loss'] for e in hist_with]
loss_wo = [e['val_loss'] for e in hist_without]
tloss_w  = [e['train_loss'] for e in hist_with]
tloss_wo = [e['train_loss'] for e in hist_without]

ax.plot(epochs_w,  tloss_w,  color='steelblue', linestyle='--', label='Train loss (aug)', linewidth=1.5)
ax.plot(epochs_w,  loss_w,   color='steelblue', linestyle='-',  label='Val loss (aug)', linewidth=2)
ax.plot(epochs_wo, tloss_wo, color='coral', linestyle='--', label='Train loss (no aug)', linewidth=1.5)
ax.plot(epochs_wo, loss_wo,  color='coral', linestyle='-',  label='Val loss (no aug)', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training vs Validation Loss')
ax.legend(fontsize=9)

plt.suptitle('Section 2: Data Augmentation — With vs. Without', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUTS / 'deep_dive_augmentation.png', bbox_inches='tight', dpi=120)
plt.show()

acc_with    = aug_with['summary']['final_test_accuracy'] * 100
acc_without = aug_without['summary']['final_test_accuracy'] * 100
print(f'\nTest Accuracy without augmentation : {acc_without:.2f}%')
print(f'Test Accuracy with    augmentation : {acc_with:.2f}%')
print(f'Delta                              : +{acc_with - acc_without:.2f}%')
print(f'\nEpoch time with aug   : {aug_with["summary"]["average_epoch_time_seconds"]:.1f}s')
print(f'Epoch time without aug: {aug_without["summary"]["average_epoch_time_seconds"]:.1f}s')

### Augmentation Analysis

**Result:** +0.10% test accuracy improvement with augmentation (99.23% vs 99.13%).  
Augmented training is ~35% slower per epoch due to on-the-fly transform computation.

**Why augmentation helps on MNIST (modestly):**
- MNIST is already very large (60K images) and low-variance — digits are centered, same scale, same orientation.
- Slight rotations and translations simulate more realistic handwriting variation.
- The model trained with augmentation generalises to slightly shifted or rotated digits at test time.

**Why the effect is small:**
- MNIST already has enough samples for the model to generalise well without augmentation.
- The augmentation used (±10°, ±10% translate, ±5% scale) is conservative — too aggressive rotation would corrupt digit identity (a rotated 6 looks like a 9).
- Larger gains from augmentation appear when datasets are small or high-variance (as seen in Part 3).

**Important note:** Augmentation is *not always beneficial* and the optimal augmentation strategy is dataset-specific. For MNIST with 60K samples, the benefit is marginal. For Part 3 with ~3.3K training images, augmentation is critical.

---
## 3. Regularization Methods Comparison

Six configurations tested on the same base architecture (cnn_medium, 421K params), 7 epochs:
- **baseline**: No regularization
- **dropout_only**: Dropout(0.3) before classifier
- **batchnorm_only**: BatchNorm after each conv layer
- **weight_decay_only**: Adam with weight_decay=1e-4 (L2 penalty on weights)
- **l1_only**: L1 penalty (λ=1e-5) added to loss
- **combined**: dropout + batchnorm + weight_decay together

In [ ]:
reg_order = ['baseline', 'dropout_only', 'batchnorm_only', 'weight_decay_only', 'l1_only', 'combined_regularization']
reg_labels = ['Baseline', 'Dropout\nonly', 'BatchNorm\nonly', 'Weight\nDecay', 'L1\nonly', 'Combined']

reg_accs  = [reg_exps[n]['summary']['final_test_accuracy'] * 100 for n in reg_order]
reg_times = [reg_exps[n]['summary']['total_training_time_seconds'] for n in reg_order]

# Print table
print(f'{'Method':<20} {'Test Acc':>9} {'Train Time':>12} {'Best Epoch':>12}')
print('-' * 58)
for name, label, acc, t in zip(reg_order, reg_labels, reg_accs, reg_times):
    best_ep = reg_exps[name]['summary']['best_epoch']
    print(f'{label.replace(chr(10), " "):<20} {acc:>8.2f}% {t:>10.1f}s {best_ep:>12}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
colors = [PALETTE[i % 10] for i in range(len(reg_order))]
bars = ax.bar(reg_labels, reg_accs, color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(reg_accs[0], color='gray', linestyle='--', linewidth=1, label=f'Baseline ({reg_accs[0]:.2f}%)')
ax.set_ylim(98.8, 99.4)
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Test Accuracy by Regularization Method')
ax.legend(fontsize=9)
for bar, acc in zip(bars, reg_accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{acc:.2f}%', ha='center', va='bottom', fontsize=8)

# Train vs val accuracy gap (overfitting indicator)
ax = axes[1]
for i, name in enumerate(reg_order):
    hist = reg_exps[name]['training_history']
    val_accs = [e['val_accuracy'] * 100 for e in hist]
    ax.plot(range(1, len(val_accs)+1), val_accs, color=colors[i],
            label=reg_labels[i].replace('\n', ' '), linewidth=1.5, marker='o', markersize=3)
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Accuracy (%)')
ax.set_title('Validation Accuracy Over Training')
ax.legend(fontsize=8, loc='lower right')

plt.suptitle('Section 3: Regularization Methods Comparison', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUTS / 'deep_dive_regularization.png', bbox_inches='tight', dpi=120)
plt.show()

### Regularization Analysis

**Ranking (test accuracy):**
1. `dropout_only` → **99.21%** (+0.12% vs baseline)
2. `batchnorm_only` → **99.13%** (+0.04%)
3. `l1_only` → **99.13%** (+0.04%)
4. `baseline` → **99.09%**
5. `weight_decay_only` → **98.99%** (−0.10%)
6. `combined` → **99.01%** (−0.08%)

**Key insights:**

- **Dropout** is the most effective single regularizer here. It forces the network to learn redundant, distributed representations — no single neuron can dominate, which reduces memorisation.

- **BatchNorm** stabilises training by normalising layer inputs — reduces internal covariate shift and acts as mild regularisation. It also allows higher learning rates and is insensitive to initialization.

- **Weight decay** (L2) underperforms here, possibly because the MNIST task is relatively easy and the model is not heavily overfit. L2 penalises large weights globally, which can be counterproductive if important features need large weight magnitudes.

- **Combined regularisation underperforms.** This is a key lesson: stacking too many regularisers doesn't always help — it can *under-regularise* by fighting against convergence. Each regularizer adds bias; combined they can make the loss landscape harder to optimise in just 7 epochs.

- **MNIST doesn't overfit much.** With 60K samples and relatively small models, the baseline already generalises well. Regularization benefits are marginal and regime-dependent.

---
## 4. Hyperparameter Tuning — 18 Configurations

Systematic sweep over: architecture, regularization, augmentation, LR, optimizer parameters, hidden size, activation function.  
All runs: 7 epochs, batch=64, Adam (unless stated), seed=42.

In [ ]:
tune_names = sorted(tune_exps.keys())

# Build results table
results = []
for name in tune_names:
    exp = tune_exps[name]
    cfg = exp['config']
    smry = exp['summary']
    results.append({
        'name': name,
        'test_acc': smry['final_test_accuracy'] * 100,
        'val_acc': smry['best_validation_accuracy'] * 100,
        'test_loss': smry['final_test_loss'],
        'params': cfg['trainable_parameters'],
        'lr': cfg['learning_rate'],
        'augmentation': cfg['augmentation_enabled'],
        'dropout': cfg['dropout'],
        'batch_norm': cfg['batch_norm'],
        'weight_decay': cfg['weight_decay'],
        'activation': cfg['activation'],
        'num_conv': cfg['num_conv_layers'],
        'train_time': smry['total_training_time_seconds'],
    })

# Sort by test accuracy
results_sorted = sorted(results, key=lambda x: x['test_acc'], reverse=True)

print(f'{'Rank':<5} {'Name':<35} {'Test Acc':>9} {'Params':>8} {'Aug':>5} {'BN':>4} {'DO':>5} {'Conv':>5}')
print('-' * 80)
for rank, r in enumerate(results_sorted, 1):
    aug = 'yes' if r['augmentation'] else 'no'
    bn  = 'yes' if r['batch_norm']   else 'no'
    print(f'{rank:<5} {r["name"]:<35} {r["test_acc"]:>8.2f}% {r["params"]:>8,} {aug:>5} {bn:>4} {r["dropout"]:>5.2f} {r["num_conv"]:>5}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# --- 1. Ranked bar chart ---
ax = axes[0, 0]
sorted_names  = [r['name'].replace('tune_', '') for r in results_sorted]
sorted_accs   = [r['test_acc'] for r in results_sorted]
bar_colors = ['gold' if i == 0 else 'steelblue' for i in range(len(sorted_names))]
bars = ax.barh(sorted_names[::-1], sorted_accs[::-1], color=bar_colors[::-1], edgecolor='black', linewidth=0.4)
ax.set_xlabel('Test Accuracy (%)')
ax.set_title('All 18 Configurations — Ranked by Test Accuracy')
ax.set_xlim(98.8, 99.5)
ax.axvline(sorted_accs[-1], color='gray', linestyle='--', linewidth=1, alpha=0.6)

# --- 2. Test accuracy vs params ---
ax = axes[0, 1]
for r in results:
    color = 'steelblue' if r['augmentation'] else 'coral'
    ax.scatter(r['params'], r['test_acc'], color=color, s=80,
               edgecolors='black', linewidths=0.4, alpha=0.8)
handles = [mpatches.Patch(color='steelblue', label='With augmentation'),
           mpatches.Patch(color='coral',     label='No augmentation')]
ax.legend(handles=handles, fontsize=9)
ax.set_xlabel('Trainable Parameters')
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Accuracy vs. Parameters (colour = augmentation)')
ax.set_ylim(98.8, 99.5)

# --- 3. Effect of augmentation ---
ax = axes[1, 0]
aug_accs   = [r['test_acc'] for r in results if r['augmentation']]
noaug_accs = [r['test_acc'] for r in results if not r['augmentation']]
ax.boxplot([noaug_accs, aug_accs], labels=['No augmentation', 'With augmentation'])
ax.scatter([1]*len(noaug_accs), noaug_accs, color='coral',     alpha=0.6, s=40, zorder=5)
ax.scatter([2]*len(aug_accs),   aug_accs,   color='steelblue', alpha=0.6, s=40, zorder=5)
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Augmentation Effect Across All Tuning Runs')

# --- 4. Learning rate effect ---
ax = axes[1, 1]
lr_accs = {}
for r in results:
    lr = r['lr']
    lr_accs.setdefault(lr, []).append(r['test_acc'])
lrs   = sorted(lr_accs.keys())
means = [np.mean(lr_accs[lr]) for lr in lrs]
ax.bar([f'LR={lr}' for lr in lrs], means, color=['steelblue', 'coral'], edgecolor='black', linewidth=0.5)
for lr, mean in zip(lrs, means):
    ax.scatter([f'LR={lr}']*len(lr_accs[lr]), lr_accs[lr], color='black', s=30, zorder=5, alpha=0.6)
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Learning Rate vs. Test Accuracy')
ax.set_ylim(98.8, 99.5)

plt.suptitle('Section 4: Hyperparameter Tuning Analysis (18 configurations)', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUTS / 'deep_dive_tuning.png', bbox_inches='tight', dpi=120)
plt.show()

### Hyperparameter Tuning Analysis

**Top 3 configurations:**
1. `tune_07_augmented` — **99.39%**: BatchNorm + Dropout(0.1) + weight_decay + **augmentation**
2. `tune_14_wide_3conv` — **99.37%**: Wide 3-conv (48/96/128) + noise + BatchNorm + Dropout(0.2)
3. `tune_10_deep_augmented` — **99.30%**: 3-conv + BatchNorm + **augmentation**

**What the sweep reveals:**

- **Augmentation is the single strongest lever.** All 3 augmentation runs (tune_07, tune_10, tune_15 area) cluster near the top. The data variance it introduces is more valuable than any other single change.

- **LR=1e-3 (standard Adam) beats LR=1e-4.** With only 7 epochs, the slow LR (tune_08) never converges as far — it needs more epochs to compensate.

- **BatchNorm + Dropout + augmentation is the winning combination.** They are complementary: augmentation diversifies data, dropout prevents co-adaptation, BatchNorm stabilises gradients.

- **Adam beta/eps tuning shows minimal effect** (tune_16, tune_17 midfield). The default Adam hyperparameters are robust for MNIST-scale problems.

- **Wider hidden layer (512) doesn't help** (tune_11: 99.18%). The bottleneck is the conv features, not the classifier capacity.

- **L1 regularization underperforms** (tune_18: 99.05%). L1 sparsifies weights aggressively which can hurt when all features are relevant.

**MLOps observation:** Without systematic tracking (experiment DB, named output folders, config.json per run), comparing 18+ runs would be nearly impossible. The folder-per-run strategy with config + summary JSON allowed this analysis to be written as pure data loading code — no manual note-taking required.

---
## 5. Best Model Deep Dive — `tune_07_augmented`

Config: 3-conv (32/64/64), BatchNorm, Dropout(0.1), weight_decay=1e-4, augmentation, LR=1e-3, 7 epochs  
Parameters: **206,346** | Test accuracy: **99.39%** | Training time: **84.9s**

In [ ]:
best_dir = OUTPUTS / 'hyperparameter_tuning_2026-04-28_155935' / 'tune_07_augmented'
best_hist = load_json(best_dir / 'training_history.json')

epochs_b    = [e['epoch'] for e in best_hist]
train_acc_b = [e['train_accuracy'] * 100 for e in best_hist]
val_acc_b   = [e['val_accuracy']   * 100 for e in best_hist]
train_loss_b = [e['train_loss'] for e in best_hist]
val_loss_b   = [e['val_loss']   for e in best_hist]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(epochs_b, train_acc_b, 'b--', label='Train accuracy', linewidth=2)
ax.plot(epochs_b, val_acc_b,   'b-',  label='Val accuracy',   linewidth=2)
ax.axvline(5, color='gold', linestyle=':', linewidth=2, label='Best epoch (5)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Best Model Training Curve (tune_07_augmented)')
ax.legend()

ax = axes[1]
ax.plot(epochs_b, train_loss_b, 'r--', label='Train loss', linewidth=2)
ax.plot(epochs_b, val_loss_b,   'r-',  label='Val loss',   linewidth=2)
ax.axvline(5, color='gold', linestyle=':', linewidth=2, label='Best epoch (5)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('Best Model Loss Curve')
ax.legend()

plt.suptitle('Section 5: Best Model Training Dynamics', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUTS / 'deep_dive_best_model_curves.png', bbox_inches='tight', dpi=120)
plt.show()

print('Training history:')
print(f'{'Epoch':<7} {'Train Acc':>10} {'Val Acc':>9} {'Train Loss':>11} {'Val Loss':>10}')
print('-' * 52)
for e in best_hist:
    marker = ' ← best' if e['epoch'] == 5 else ''
    print(f'{e["epoch"]:<7} {e["train_accuracy"]*100:>9.2f}% {e["val_accuracy"]*100:>8.2f}% '
          f'{e["train_loss"]:>11.5f} {e["val_loss"]:>10.5f}{marker}')

In [ ]:
# Confusion matrix of best model
cm_path = best_dir / 'confusion_matrix.png'
if cm_path.exists():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(plt.imread(str(cm_path)))
    axes[0].set_title('Confusion Matrix — tune_07_augmented (99.39%)', fontsize=11)
    axes[0].axis('off')

    incorrect_path = best_dir / 'incorrect_predictions.png'
    if incorrect_path.exists():
        axes[1].imshow(plt.imread(str(incorrect_path)))
        axes[1].set_title('Sample Incorrect Predictions', fontsize=11)
        axes[1].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Confusion matrix image not found.')

In [ ]:
# Conv filter visualizations of best model
f1 = best_dir / 'conv_filters_first.png'
f2 = best_dir / 'conv_filters_last.png'

if f1.exists() and f2.exists():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(plt.imread(str(f1)))
    axes[0].set_title('Layer 1 Conv Filters\n(edges, gradients, strokes)', fontsize=11)
    axes[0].axis('off')
    axes[1].imshow(plt.imread(str(f2)))
    axes[1].set_title('Layer 3 Conv Filters\n(complex digit parts, curves, junctions)', fontsize=11)
    axes[1].axis('off')
    plt.suptitle('Learned Filters: tune_07_augmented', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print('Filter images not found.')

### Best Model Analysis

**Checkpoint strategy and overfitting:**
- Best validation accuracy occurs at **epoch 5** of 7 (validation loss = 0.0266).
- After epoch 5, validation loss increases slightly while training accuracy continues rising — classic mild overfitting onset.
- **The checkpoint at epoch 5 is what gives us 99.39% test accuracy**, not the final epoch. This validates the checkpoint-saving strategy: saving every N epochs and choosing the best by validation loss, not just taking the last checkpoint.

**Confusion matrix observations:**
- The most confused digit pairs on MNIST are typically **4↔9**, **3↔5**, **7↔1**.
- With 99.39% accuracy = ~61 errors on 10,000 test images.
- Errors cluster at visually similar digits — especially when handwriting quality is poor.

**Conv filter progression:**
- **Layer 1 filters** resemble oriented Gabor filters: detect horizontal, vertical, and diagonal strokes at different phases. These are universal low-level visual features — they look nearly the same across different CNN architectures.
- **Layer 3 filters** are higher-dimensional and harder to interpret visually. They respond to combinations of strokes that encode meaningful digit structure: loops, crossings, serifs, endpoints.

This hierarchy — from edges to parts to wholes — is the fundamental representational power of CNNs, and why they outperform FFNs for image classification.

---
## 6. MLOps Practices Summary

The Part 2 experiments implemented the following MLOps practices:

| Practice | Implementation |
|---|---|
| **Run versioning** | Timestamped output folders per run |
| **Config tracking** | `config.json` saved with every run |
| **Checkpoint saving** | `best_model.pt` + periodic `checkpoint_epoch_N.pt` |
| **Best model selection** | Saved by best *validation* accuracy, not last epoch |
| **Experiment database** | SQLite `experiments.db` logs all runs |
| **Performance metrics** | Epoch time, total time, time-to-best-model |
| **Training visualisation** | Loss + accuracy curves saved as PNG per run |
| **Error analysis** | Confusion matrix + correct/incorrect prediction grids |
| **Filter visualisation** | `conv_filters_first/last.png` per run |
| **Git tracking** | `git_commit` + `git_is_dirty` logged in config |

**Advanced MLOps tools** (not implemented here but relevant):  
- **Weights & Biases** (wandb): Real-time dashboard, hyperparameter sweep UI, model registry  
- **MLflow**: Experiment tracking, model versioning, deployment  
- **DVC**: Data version control alongside code  
- **Optuna / Ray Tune**: Automated hyperparameter optimisation with pruning of bad runs

The folder-per-run + JSON approach used here is a lightweight but effective alternative for small-scale research.

---
## 7. Conclusions

### MNIST CNN — What We Learned

**Architecture:**
- 3-conv layers > 2-conv layers for the same parameter budget.
- Channel depth matters more than classifier width.
- Max-pooling after each conv provides translation invariance — critical for digit recognition.

**Regularisation:**
- Dropout is most effective for MNIST. BatchNorm adds stability.
- Stacking all regularisers doesn't compound their benefits — can hurt if over-applied in short training.
- MNIST needs less regularisation than real-world datasets due to its simplicity and scale.

**Augmentation:**
- Small but consistent gain (+0.10%). More impactful on small or high-variance datasets.
- Conservative transforms are safer: avoid transforms that change digit identity (heavy rotation).

**Hyperparameter tuning:**
- Augmentation + BatchNorm + light Dropout = best combo.
- LR=1e-3 (Adam default) is right for 7 epochs; lower LR needs more epochs to be useful.
- Best accuracy achieved: **99.39%** with 206K parameters and <90s training time.

**Best model for deployment:** `tune_07_augmented` or `tune_14_wide_3conv`  
Both achieve top accuracy with compact models (206K–451K params) and fast inference.